<a href="https://colab.research.google.com/github/mayajdias/ds2002-fa26/blob/main/notebooks/01/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** a single line item on a single order (one item, with its quantity, from one order), with the order's `order` and `vendor_id` carried along.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
vendors = pd.DataFrame(vendors_json)

# validate='m:1' fails loudly if vendors had duplicate vendor_ids (which would silently multiply rows)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True, validate='m:1')

unmatched = joined[joined['_merge'] == 'left_only']
print("Unmatched vendor(s):", unmatched['vendor_id'].unique().tolist())
print("Units attached to unmatched vendor(s):", unmatched['qty'].sum())
joined

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
# Keep the unmatched vendor as its own labeled row instead of dropping it. A default groupby
# silently discards NaN keys, so the totals would add up to 4 instead of 7 and the missing
# units would be invisible. Labeling it keeps the table reconciled to 7 and flags the data problem.
joined['name_clean'] = joined['name'].fillna('UNMATCHED (' + joined['vendor_id'] + ')')

qty_by_vendor = joined.groupby('name_clean')['qty'].sum().sort_values(ascending=False)
print(qty_by_vendor)
assert qty_by_vendor.sum() == 7

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
# V-08 has no zone, so its 3 units get their own "No zone" bucket rather than vanishing.
joined['zone_clean'] = joined['zone'].fillna('No zone (unmatched vendor)')

qty_by_zone = joined.groupby('zone_clean')['qty'].sum().sort_values(ascending=False)
print(qty_by_zone)

matched_total = qty_by_zone.drop('No zone (unmatched vendor)').sum()
print(f"\nTop zone: {qty_by_zone.index[0]} with {qty_by_zone.iloc[0]} items")
print(f"Zone total counting only real zones: {matched_total} (not 7)")
print(f"Grand total including the unmatched bucket: {qty_by_zone.sum()}")
print("The 3-unit difference is the V-08 Hot Dog order, which has no zone because V-08 is not in the vendor list.")

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO

### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_your answer here_